# Northstar Triage Copilot v2 — loads the trained teammate models

This app **trains nothing**. It loads the six model bundles saved by the training
notebooks and serves them behind two tabs.

| Shown as | Model | Trained in |
|---|---|---|
| Case type | M1 | `train_m1_case_type.ipynb` (Louis — summary only, group split) |
| Category | M2 | `train_m2_category.ipynb` (Category nb — summary only, group split) |
| Priority — direct | M3 | `train_m3_priority_tracy.ipynb` (Tracy — direct, summary+plan+tenure+channel) |
| Priority — check | M3 | `train_m3_priority_milind.ipynb` (Milind — double-check, no proxies) |
| Suggested route | M5 | `train_m5_route.ipynb` (Christian — summary only, review gate) |
| Escalation risk | M4 | `train_m4_escalation.ipynb` (Anna — tuned threshold, time split) |

**Two priority models by design.** Tracy predicts the priority directly (and uses
`plan_tier`/`tenure`); Milind refuses to trust an exact prediction and instead
flags cases where the agent's priority is ≥2 levels from his estimate. The card
shows both and marks when they disagree.

The Triage tab runs a **cascade**: case type and category are predicted from the
summary, then feed the priority and escalation models.

## 1. Load the saved bundles (no training here)

In [ ]:
import os
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import json as _json, html, warnings
import numpy as np, pandas as pd, joblib
warnings.filterwarnings("ignore")

KEYS = ["m1_case_type","m2_category","m3_priority_tracy","m3_priority_milind","m4_escalation","m5_route"]
B = {}
for k in KEYS:
    p = f"models/{k}.joblib"
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} missing — run train_{k}.ipynb first")
    B[k] = joblib.load(p)
ORDER = ["m1_case_type","m2_category","m3_priority_tracy","m3_priority_milind","m5_route","m4_escalation"]
for k in ORDER:
    print(f'{k:20s} {B[k]["label"]:24s} {B[k]["meta"].get("author",""):10s} {B[k]["metrics"]}')

## 2. Prediction cascade

In [ ]:
PRIORITIES = B["m3_priority_milind"]["meta"]["priorities"]
RANK = {p:i for i,p in enumerate(PRIORITIES)}

def _row(base, extra=None):
    return pd.DataFrame([{**base, **(extra or {})}])

def _proba_order(key, row):
    b = B[key]; m = b["model"]; p = m.predict_proba(row[b["features"]])[0]
    order = p.argsort()[::-1]
    return [(str(m.classes_[i]), float(p[i])) for i in order], m, p

def predict_case(summary, channel, plan, month, tenure, tags, agent_priority=None):
    tags = (tags or "").strip()
    base = {"case_summary": summary or "", "tags": tags, "channel": channel, "plan_tier": plan,
            "month_num": int(month), "customer_tenure_months": int(tenure),
            "n_tags": 0 if not tags else len(tags.split(";"))}

    ct,_,_   = _proba_order("m1_case_type", _row(base))
    cat,_,_  = _proba_order("m2_category",  _row(base))
    case_type, category = ct[0][0], cat[0][0]

    # two priority models: Tracy = direct predictor (uses plan/tenure), Milind = double-check
    pri_direct,_,_ = _proba_order("m3_priority_tracy",  _row(base))
    pri_check,_,_  = _proba_order("m3_priority_milind", _row(base, {"case_type":case_type}))

    rt,_,_ = _proba_order("m5_route", _row(base))
    b4 = B["m4_escalation"]; m4 = b4["model"]
    p4 = m4.predict_proba(_row(base, {"category":category})[b4["features"]])[0]
    esc_p = float(p4[list(m4.classes_).index(True)]); esc_flag = esc_p >= b4["threshold"]

    review = rt[0][1] < B["m5_route"].get("review_threshold", 0.0)
    out = {"case_type":ct[0], "category":cat[0], "route":rt[0],
           "priority_direct":pri_direct[0], "priority_check":pri_check[0],
           "pri_agree": pri_direct[0][0]==pri_check[0][0],
           "esc_prob":esc_p, "esc_flag":esc_flag, "review":review,
           "terms":_top_terms("m1_case_type", _row(base), 3)}
    if agent_priority in RANK:
        gap = abs(RANK[agent_priority] - RANK[pri_check[0][0]])   # double-check uses Milind
        out["priority_flag"] = gap >= B["m3_priority_milind"]["meta"]["flag_gap"]
        out["agent_priority"] = agent_priority
    return out

_STOP = set("the a an and or of to in for with on at is are be was were this that it its my our your "
            "i we you they he she as by from not no cannot can keeps after getting get got need want "
            "please thanks cheers hi hello there their them has have had do does did been being but so "
            "if then when which who what why how able keep still just also very more most".split())
def _top_terms(key, row, k=3):
    b = B[key]; m = b["model"]
    prep, clf = m.named_steps["prep"], m.named_steps["clf"]
    names = prep.get_feature_names_out()
    x = prep.transform(row[b["features"]]); xr = np.asarray(x.todense()).ravel()
    ci = list(clf.classes_).index(m.predict(row[b["features"]])[0])
    coef = clf.coef_; c = coef[ci] if coef.shape[0] > 1 else (coef[0] if ci==1 else -coef[0])
    contrib = c * xr
    pref = "case_summary_w__"
    idx = [i for i in range(len(names)) if names[i].startswith(pref)]
    idx.sort(key=lambda i: contrib[i], reverse=True)
    out=[]
    for i in idx:
        if contrib[i] <= 0: break
        term=" ".join(w for w in names[i].replace(pref,"").split() if w not in _STOP and len(w)>2)
        if term and term not in out: out.append(term)
        if len(out)>=k: break
    return out

print(predict_case("Cannot connect the Xero integration, OAuth token fails with a 500 error",
                    "email", "Enterprise", 6, 30, "xero;integration"))

## 3. Result card (Triage tab)

In [ ]:
THR = B["m4_escalation"]["threshold"]
def band(p, thr=THR):
    if p>=thr:     return "High","#c0392b","#fdecea"
    if p>=thr/2:   return "Medium","#b9770e","#fef5e7"
    return "Low","#1e8449","#eafaf1"

def explain(o):
    bits=[]
    if o["terms"]: bits.append(", ".join(o["terms"]) + " in summary")
    bits.append(f'{o["case_type"][0].replace("_"," ")} language')
    if o["esc_flag"]:
        bits.append("flagged above the tuned escalation threshold")
    elif o["esc_prob"]>=THR/2:
        bits.append("elevated but below the escalation threshold")
    else:
        bits.append("below the escalation threshold")
    return " + ".join(bits)

def render_card(o):
    lv,col,bg = band(o["esc_prob"])
    agree = '' if o["pri_agree"] else ' <span class="rev">models differ</span>'
    rows=[("Case type", f'{o["case_type"][0].replace("_"," ").title()} &mdash; {o["case_type"][1]:.0%} confidence'),
          ("Category",  f'{o["category"][0].replace("_"," ").title()} &mdash; {o["category"][1]:.0%}'),
          ("Priority — direct (Tracy)", f'{o["priority_direct"][0]} &mdash; {o["priority_direct"][1]:.0%}'),
          ("Priority — check (Milind)", f'{o["priority_check"][0]} &mdash; {o["priority_check"][1]:.0%}{agree}'),
          ("Suggested route", o["route"][0].title() + (
               ' <span class="rev">needs human review</span>' if o["review"] else ''))]
    tr="".join(f'<tr><td class="k">{k}</td><td class="v">{v}</td></tr>' for k,v in rows)
    risk=(f'<tr><td class="k">Risk</td><td class="v"><span class="pill" '
          f'style="color:{col};background:{bg}">{lv} escalation risk &mdash; {o["esc_prob"]:.0%}</span></td></tr>')
    extra=""
    if o.get("priority_flag"):
        extra=(f'<tr><td class="k">Priority check</td><td class="v"><span class="pill" '
               f'style="color:#c0392b;background:#fdecea">review: you set '
               f'{o["agent_priority"]}, Milind estimates {o["priority_check"][0]}</span></td></tr>')
    why=f'<tr><td class="k">Why</td><td class="v why">{html.escape(explain(o))}</td></tr>'
    return f'<div class="card"><table class="res">{tr}{risk}{extra}{why}</table></div>'
print(render_card(predict_case("Dashboard is down for all users, 500 errors everywhere","phone","Pro",9,36,"outage"))[:90],"...")

## 4. Tab 2 — model cards, held-out inspector, metrics

In [ ]:
def model_cards():
    g=""
    for k in ORDER:
        b=B[k]; m=b["metrics"]; meta=b["meta"]
        met=" · ".join(f'{kk} <b>{vv}</b>' for kk,vv in m.items() if kk!="baseline_acc")
        g+=(f'<div class="mcard"><div class="mh">{k.split("_")[0].upper()} — {b["label"]}</div>'
            f'<div class="mb">'
            f'<div class="spec"><span>Author</span>{meta.get("author","")}</div>'
            f'<div class="spec"><span>Algorithm</span>Logistic Regression (class-weighted) on TF-IDF + structured</div>'
            f'<div class="spec"><span>Inputs</span>{meta.get("inputs","")}</div>'
            f'<div class="spec"><span>Split</span>{meta.get("split","")}</div>'
            f'<div class="spec"><span>Classes</span>{len(b["classes"])}</div>'
            f'<div class="spec"><span>Held-out</span>{met}</div>'
            + (f'<div class="cav">{meta.get("note","")}</div>' if meta.get("note") else "")
            + '</div></div>')
    return f'<div class="mgrid">{g}</div>'

def metrics_table():
    rows=[]
    for k in ORDER:
        b=B[k]; r={"Model":k.split("_")[0].upper(),"Predicts":b["label"]}
        r.update({kk:vv for kk,vv in b["metrics"].items()})
        rows.append(r)
    return pd.DataFrame(rows)

def case_choices(key):
    t=B[key]["test"]
    return [(f'{r.case_id} — {str(r.case_summary)[:55]}', i) for i,r in t.iterrows()]

def inspect(key, idx):
    if key is None or idx is None: return '<div class="card"><p class="note">Pick a model and a case.</p></div>'
    b=B[key]; t=b["test"]; row=t.iloc[int(idx)]
    actual=row[b["target"]]; pred=row["predicted"]; conf=row.get("confidence",None)
    ok = str(pred)==str(actual)
    mark='<span style="color:#1e8449">&#10004; match</span>' if ok else '<span style="color:#c0392b">&#10008; differs</span>'
    extra = f' · prob(escalate) {row["prob"]:.0%} vs threshold {b.get("threshold",0):.0%}' if b["kind"]=="binary" else ""
    return (f'<div class="card"><p class="sum">&ldquo;{html.escape(str(row.case_summary)[:300])}&rdquo;</p>'
            f'<table class="insp"><tr><th>Field</th><th>Value</th></tr>'
            f'<tr><td>Model</td><td>{key.split("_")[0].upper()} — {b["label"]}</td></tr>'
            f'<tr><td>Predicted</td><td>{html.escape(str(pred))}'
            + (f' ({conf:.0%})' if conf is not None else '') + f'{extra}</td></tr>'
            f'<tr><td>Actual</td><td>{html.escape(str(actual))}</td></tr>'
            f'<tr><td>Result</td><td>{mark}</td></tr></table>'
            f'<p class="note">From the held-out split of {B[key]["meta"].get("split","")}; the model never trained on this case.</p></div>')
print("tab-2 helpers ready |", metrics_table().shape)

## 5. The app (loader only)

In [ ]:
import gradio as gr
CHANNELS=["email","webchat","in_app","phone"]
PLANS=["Free","Standard","Pro","Enterprise"]
LABELS={k:f'{k.split("_")[0].upper()} — {B[k]["label"]}' for k in ORDER}
NB_BY_KEY={k:os.path.abspath(f"train_{k}.ipynb") for k in KEYS}
EXAMPLE=["Cannot connect the Xero integration — the OAuth token keeps failing with a 500 error after the latest update",
         "in_app","Enterprise",6,30,"xero;integration;error","(none)"]

CSS="""
#app {max-width:1060px;margin:auto;}
.card{border:1px solid #e3e6ea;border-radius:14px;padding:22px 26px;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.05);}
table.res{width:100%;border-collapse:collapse;}
table.res td{padding:12px 6px;border-bottom:1px solid #f0f1f3;font-size:16px;vertical-align:top;}
table.res tr:last-child td{border-bottom:none;}
td.k{color:#6b7280;width:190px;font-weight:600;} td.v{color:#111827;} td.v.why{color:#374151;}
.pill{padding:4px 12px;border-radius:999px;font-weight:700;font-size:14px;}
.rev{color:#b9770e;background:#fef5e7;padding:2px 8px;border-radius:999px;font-size:12px;font-weight:700;}
.sum{font-style:italic;color:#374151;margin:0 0 14px 0;}
table.insp{width:100%;border-collapse:collapse;}
table.insp th{text-align:left;color:#6b7280;font-size:13px;border-bottom:2px solid #eef0f2;padding:6px;}
table.insp td{padding:8px 6px;border-bottom:1px solid #f3f4f6;font-size:14px;}
.note{color:#9ca3af;font-size:12px;margin-top:12px;}
.mgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:14px;}
.mcard{border:1px solid #e3e6ea;border-radius:12px;overflow:hidden;background:#fff;}
.mh{background:#1f2937;color:#fff;padding:10px 14px;font-weight:700;font-size:14px;}
.mb{padding:12px 14px;font-size:13px;color:#111827;line-height:1.5;}
.spec{margin-bottom:6px;} .spec>span{display:inline-block;width:86px;color:#6b7280;font-weight:600;vertical-align:top;}
.cav{color:#8a5a00;font-size:12px;margin-top:8px;padding-top:8px;border-top:1px dashed #eef0f2;}
"""

with gr.Blocks(title="Northstar Triage Copilot v2", css=CSS, theme=gr.themes.Soft()) as demo:
    with gr.Column(elem_id="app"):
        gr.Markdown("# Northstar Triage Copilot\nFive trained models, loaded from disk. One screen for the agent, one for trust.")
        with gr.Tabs():
            with gr.Tab("Triage a case"):
                with gr.Row():
                    with gr.Column(scale=5):
                        summary=gr.Textbox(label="Case summary", lines=4, placeholder="Paste the customer's case summary…")
                        with gr.Row():
                            channel=gr.Dropdown(CHANNELS, value="email", label="Channel")
                            plan=gr.Dropdown(PLANS, value="Standard", label="Plan tier (Tracy's model only)")
                        with gr.Row():
                            month=gr.Dropdown(list(range(1,13)), value=6, label="Month")
                            tenure=gr.Number(value=12, label="Tenure (months)")
                        agent_pri=gr.Dropdown(["(none)"]+PRIORITIES, value="(none)", label="Agent's priority (optional, for Milind's double-check)")
                        tags=gr.Textbox(label="Tags (optional, ';'-separated)", placeholder="leave blank if unknown")
                        with gr.Row():
                            go=gr.Button("Triage case", variant="primary"); ex=gr.Button("Load example")
                    with gr.Column(scale=5):
                        out=gr.HTML('<div class="card"><p class="note">Enter a case and press <b>Triage case</b>, or <b>Load example</b>.</p></div>')
                def triage(s,c,pl,mo,tn,tg,ap):
                    return render_card(predict_case(s,c,pl,mo,tn,tg,None if ap=="(none)" else ap))
                ins=[summary,channel,plan,month,tenure,tags,agent_pri]
                go.click(triage, ins, out)
                ex.click(lambda: EXAMPLE, None, ins).then(triage, ins, out)

            with gr.Tab("How the models work"):
                gr.Markdown("### A. Model cards")
                gr.HTML(model_cards())
                gr.Markdown("### B. Inspect a held-out case (predicted vs actual)")
                with gr.Row():
                    mdl=gr.Dropdown([(LABELS[k],k) for k in ORDER], label="Model")
                    case=gr.Dropdown([], label="Held-out case")
                insp=gr.HTML('<div class="card"><p class="note">Pick a model, then a case.</p></div>')
                mdl.change(lambda k: gr.update(choices=case_choices(k), value=None), mdl, case)
                case.change(inspect, [mdl, case], insp)
                gr.Markdown("### C. Held-out metrics (as saved by the training notebooks)")
                gr.Dataframe(metrics_table(), interactive=False)
                gr.Markdown("### D. Training notebooks")
                gr.Markdown("Each model is trained in its own notebook (imports `northstar_prep.py`). Download to open:")
                with gr.Row():
                    for k in ORDER:
                        gr.DownloadButton(f"⬇ train_{k}", value=NB_BY_KEY[k], size="sm")

print("app v2 built | loads", len(B), "models")

## 6. Launch

In [ ]:
# demo.launch(server_name="127.0.0.1", server_port=7866, inbrowser=False)